# Module 18: Data Engineering Basics

**Lesson: Building Data Pipelines for ML Features**

This notebook covers ETL/ELT patterns, working with APIs, data formats (Parquet, Avro, Arrow), Apache Airflow, data quality checks, cloud storage, and pipeline monitoring.

In [ ]:
import json
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import io
print('Data engineering libraries loaded')

## 1. ETL vs ELT Patterns

ETL (Extract, Transform, Load) transforms data before loading. ELT (Extract, Load, Transform) loads raw data first, then transforms. ELT suits big data and data lakes.

In [ ]:
# ETL Pattern: Transform before loading to target
def etl_extract():
    return pd.DataFrame({
        'raw_date': ['2024-01-01', '2024-01-02', '2024-01-03'],
        'raw_price': ['$10.50', '$20.30', '$15.00'],
        'raw_quantity': ['5', '10', '7']
    })

def etl_transform(df):
    df = df.copy()
    df['date'] = pd.to_datetime(df['raw_date'])
    df['price'] = df['raw_price'].str.replace('$', '').astype(float)
    df['quantity'] = df['raw_quantity'].astype(int)
    df['total'] = df['price'] * df['quantity']
    return df[['date', 'price', 'quantity', 'total']]

def etl_load(df):
    # In real scenario: write to database or data warehouse
    return f'Loaded {len(df)} rows to target'

# Run ETL
raw = etl_extract()
transformed = etl_transform(raw)
result = etl_load(transformed)
print('ETL Pattern:')
print(result)
print('\nTransformed data:')
print(transformed)

print('\n---\n')

# ELT Pattern: Load raw data first
def elt_load():
    return etl_extract()  # Load raw to data lake

def elt_transform(df):
    return etl_transform(df)  # Transform in place

raw = elt_load()
print('ELT Pattern: Raw data loaded to data lake')
print('Transform happens on read/query time')

## 2. Working with APIs (requests library)

Extracting data from REST APIs is fundamental to data engineering. Key concerns: authentication, pagination, rate limiting, and error handling.

In [ ]:
import requests
import time

# Simulated API client with rate limiting and retry logic
class APIClient:
    def __init__(self, base_url='https://api.example.com', max_calls=10, period=60):
        self.base_url = base_url
        self.max_calls = max_calls
        self.period = period
        self.call_times = []
    
    def _rate_limit(self):
        now = time.time()
        self.call_times = [t for t in self.call_times if t > now - self.period]
        if len(self.call_times) >= self.max_calls:
            sleep_time = self.call_times[0] + self.period - now
            if sleep_time > 0:
                print(f'Rate limited: sleeping {sleep_time:.1f}s')
                time.sleep(sleep_time)
        self.call_times.append(time.time())
    
    def fetch_with_retry(self, endpoint, max_retries=3):
        for attempt in range(max_retries):
            try:
                self._rate_limit()
                url = f'{self.base_url}/{endpoint}'
                # Simulate successful response
                if attempt == 0:
                    return {'status': 'ok', 'data': [{'id': 1, 'value': 'sample'}]}
            except requests.RequestException as e:
                print(f'Attempt {attempt + 1} failed: {e}')
                if attempt == max_retries - 1:
                    raise
                time.sleep(2 ** attempt)
        return None

client = APIClient()
response = client.fetch_with_retry('data')
print('API response:', response['status'])
print(f'Found {len(response["data"])} records')

print('\nKey concepts:')
print('- Rate limiting: {:.0f} calls per {:.0f}s window'.format(client.max_calls, client.period))
print('- Exponential backoff: 1s, 2s, 4s wait between retries')

## 3. Data Formats: Parquet, Avro, Arrow

Modern data engineering uses columnar formats (Parquet, Arrow) for analytics and row-based formats (Avro) for streaming and schema evolution.

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import fastavro

# Create sample data
df = pd.DataFrame({
    'feature_1': np.random.randn(1000),
    'feature_2': np.random.randn(1000),
    'target': np.random.randint(0, 2, 1000),
    'category': np.random.choice(['A', 'B', 'C'], 1000),
    'timestamp': pd.date_range('2024-01-01', periods=1000, freq='h')
})

# Parquet: columnar, compressed, efficient for analytics
buf = io.BytesIO()
df.to_parquet(buf, compression='snappy')
buf.seek(0)
parquet_size = buf.tell()
df_parquet = pd.read_parquet(buf)

# CSV for comparison
csv_buf = io.BytesIO()
df.to_csv(csv_buf)
csv_size = csv_buf.tell()

print('=== Data Format Comparison ===')
print(f'CSV size:     {csv_size:,} bytes')
print(f'Parquet size: {parquet_size:,} bytes (snappy compressed)')
print(f'Parquet is {csv_size / parquet_size:.1f}x smaller')
print(f'\nParquet is the standard for ML feature stores')
print(f'Avro is used for streaming and Kafka')
print(f'Arrow is used for in-memory analytics and zero-copy sharing')

In [ ]:
# Avro: row-based, schema evolution, good for streaming
avro_schema = {
    'type': 'record',
    'name': 'MLEvent',
    'fields': [
        {'name': 'event_id', 'type': 'string'},
        {'name': 'timestamp', 'type': 'long'},
        {'name': 'feature_name', 'type': 'string'},
        {'name': 'feature_value', 'type': 'double'},
        {'name': 'model_version', 'type': ['null', 'string'], 'default': None}
    ]
}

# Write Avro
records = [
    {'event_id': 'e1', 'timestamp': int(time.time()), 'feature_name': 'f1', 'feature_value': 0.5},
    {'event_id': 'e2', 'timestamp': int(time.time()), 'feature_name': 'f2', 'feature_value': 0.8, 'model_version': 'v2'}
]

avro_buf = io.BytesIO()
fastavro.writer(avro_buf, avro_schema, records)
avro_buf.seek(0)
avro_reader = fastavro.reader(avro_buf)
for record in avro_reader:
    print('Avro record:', record)

print('\nKey Avro features:')
print('- Schema evolution: add/remove fields with defaults')
print('- Row-oriented: efficient for writes and streaming')
print('- Compact binary format')

## 4. Apache Airflow Concepts

Airflow orchestrates complex data pipelines as directed acyclic graphs (DAGs). Key concepts: DAG, Task, Operator, Sensor, Schedule.

In [ ]:
# Simulating Airflow concepts without the actual Airflow runtime
# These patterns mirror what Airflow DAGs look like

class Task:
    """Simulate an Airflow task"""
    def __init__(self, name, func, dependencies=None):
        self.name = name
        self.func = func
        self.dependencies = dependencies or []
        self.output = None
    
    def execute(self, context):
        print(f'[Task: {self.name}] Starting...')
        self.output = self.func(context)
        print(f'[Task: {self.name}] Completed')
        return self.output

class DAG:
    """Simulate an Airflow DAG"""
    def __init__(self, dag_id, schedule=None):
        self.dag_id = dag_id
        self.schedule = schedule
        self.tasks = {}
    
    def add_task(self, task):
        self.tasks[task.name] = task
    
    def run(self, context=None):
        print(f'\n=== Running DAG: {self.dag_id} ===')
        context = context or {}
        # Topological sort and execute
        executed = set()
        while len(executed) < len(self.tasks):
            for name, task in self.tasks.items():
                if name not in executed:
                    deps_met = all(d in executed for d in task.dependencies)
                    if deps_met:
                        task.execute(context)
                        executed.add(name)
        print(f'=== DAG {self.dag_id} Complete ===')

print('Airflow concepts:')
print('- DAG: Directed Acyclic Graph (pipeline definition)')
print('- Task: Unit of work within a DAG')
print('- Operator: Template for tasks (PythonOperator, BashOperator)')
print('- Sensor: Waits for external events')
print('- Schedule: cron expression for execution frequency')

In [ ]:
# Build a simulated ML feature pipeline DAG
def extract_features(context):
    print('  Extracting raw data from API...')
    context['raw_data'] = {'rows': 1000, 'columns': 15}
    return context['raw_data']

def validate_raw(context):
    print('  Validating raw data...')
    raw = context.get('raw_data', {})
    assert raw.get('rows', 0) > 0, 'No data extracted'
    print(f'  Raw data OK: {raw["rows"]} rows')
    return 'validated'

def transform_features(context):
    print('  Engineering features...')
    context['features'] = {'columns': ['f1', 'f2', 'f3'], 'rows': 1000}
    return context['features']

def validate_features(context):
    print('  Checking feature quality...')
    features = context.get('features', {})
    print(f'  Features OK: {len(features["columns"])} columns')
    return 'quality_ok'

def upload_features(context):
    print('  Uploading features to S3...')
    print('  Upload complete: features/2024/01/01/features.parquet')
    return 'uploaded'

# Assemble the DAG
ml_dag = DAG('ml_feature_pipeline', schedule='@daily')

t_extract = Task('extract', extract_features)
t_validate_raw = Task('validate_raw', validate_raw, dependencies=['extract'])
t_transform = Task('transform', transform_features, dependencies=['validate_raw'])
t_validate_feat = Task('validate_features', validate_features, dependencies=['transform'])
t_upload = Task('upload', upload_features, dependencies=['validate_features'])

for t in [t_extract, t_validate_raw, t_transform, t_validate_feat, t_upload]:
    ml_dag.add_task(t)

# Run the DAG
ml_dag.run()

print('\nIn Airflow, dependencies are chained: extract >> validate_raw >> transform >> validate_features >> upload')

## 5. Data Quality Checks with Great Expectations

Data quality is critical for ML pipelines. Great Expectations provides a framework for defining, running, and documenting data quality expectations.

In [ ]:
# Simulating Great Expectations checks (simplified version)
class ExpectationSuite:
    def __init__(self, name):
        self.name = name
        self.expectations = []
    
    def add_expectation(self, expectation_func):
        self.expectations.append(expectation_func)
    
    def validate(self, df):
        results = []
        all_passed = True
        for exp in self.expectations:
            passed, msg = exp(df)
            results.append({'passed': passed, 'message': msg})
            if not passed:
                all_passed = False
        return {'success': all_passed, 'results': results}

# Define expectations
def expect_no_nulls(column):
    def check(df):
        nulls = df[column].isna().sum()
        return nulls == 0, f'{column}: {nulls} null values'
    return check

def expect_column_between(column, min_val, max_val):
    def check(df):
        col = df[column]
        violations = ((col < min_val) | (col > max_val)).sum()
        return violations == 0, f'{column}: {violations} values outside [{min_val}, {max_val}]'
    return check

def expect_row_count_between(min_rows, max_rows):
    def check(df):
        n = len(df)
        return min_rows <= n <= max_rows, f'Row count: {n} (expected [{min_rows}, {max_rows}])'
    return check

# Create quality suite for ML data
suite = ExpectationSuite('ml_feature_suite')
suite.add_expectation(expect_no_nulls('feature_1'))
suite.add_expectation(expect_no_nulls('target'))
suite.add_expectation(expect_column_between('feature_1', -5, 5))
suite.add_expectation(expect_column_between('target', 0, 1))
suite.add_expectation(expect_row_count_between(100, 10000))

# Validate
result = suite.validate(df)
print('=== Data Quality Validation ===')
print(f'Suite: {suite.name}')
print(f'Overall: {"PASSED" if result["success"] else "FAILED"}')
for r in result['results']:
    status = 'PASS' if r['passed'] else 'FAIL'
    print(f'  [{status}] {r["message"]}')

## 6. Cloud Storage with boto3 (S3 Mock)

S3-compatible storage is standard for ML data lakes. Use moto for testing without real AWS.

In [ ]:
import boto3
from moto import mock_s3

@mock_s3
def demo_s3_operations():
    client = boto3.client('s3', region_name='us-east-1')
    
    # Create bucket
    client.create_bucket(Bucket='ml-features')
    print('Bucket created: ml-features')
    
    # Upload data (simulate Parquet file)
    buf = io.BytesIO()
    df.to_parquet(buf, compression='snappy')
    buf.seek(0)
    
    key = 'features/2024/01/01/feature_set_v1.parquet'
    client.put_object(
        Bucket='ml-features',
        Key=key,
        Body=buf.getvalue()
    )
    print(f'Uploaded: {key}')
    
    # List objects
    response = client.list_objects_v2(Bucket='ml-features')
    print(f'Objects in bucket:')
    for obj in response.get('Contents', []):
        print(f'  - {obj["Key"]} ({obj["Size"]:,} bytes)')
    
    # Download and read
    response = client.get_object(Bucket='ml-features', Key=key)
    downloaded = pd.read_parquet(io.BytesIO(response['Body'].read()))
    print(f'\nDownloaded Parquet: {len(downloaded)} rows x {len(downloaded.columns)} cols')
    
    # Cleanup
    client.delete_object(Bucket='ml-features', Key=key)
    client.delete_bucket(Bucket='ml-features')
    print('Cleaned up test bucket')

demo_s3_operations()

## 7. Pipeline Monitoring

Monitoring ensures pipeline reliability. Track run status, data volumes, timing, and set up alerts on failures.

In [ ]:
import logging
import time as time_module

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(name)s | %(levelname)s | %(message)s')
logger = logging.getLogger('pipeline')

class PipelineMonitor:
    def __init__(self, pipeline_name):
        self.pipeline_name = pipeline_name
        self.start_time = None
        self.metrics = {}
    
    def start_run(self):
        self.start_time = time_module.time()
        logger.info(f'Pipeline "{self.pipeline_name}" started')
    
    def log_step(self, step_name, row_count=None, status='success'):
        elapsed = time_module.time() - self.start_time
        self.metrics[step_name] = {'row_count': row_count, 'status': status, 'elapsed': round(elapsed, 2)}
        logger.info(f'Step "{step_name}": {"OK" if status == "success" else "FAILED"} '
                    f'(rows={row_count}, elapsed={elapsed:.1f}s)')
    
    def end_run(self, status='success'):
        total_time = time_module.time() - self.start_time
        logger.info(f'Pipeline "{self.pipeline_name}" finished: status={status}, total={total_time:.1f}s')
        return {'pipeline': self.pipeline_name, 'status': status, 'total_time': total_time, 'metrics': self.metrics}

# Demo monitoring
monitor = PipelineMonitor('ml_feature_pipeline')
monitor.start_run()
time_module.sleep(0.1)
monitor.log_step('extract', row_count=1000)
time_module.sleep(0.1)
monitor.log_step('transform', row_count=1000)
time_module.sleep(0.1)
monitor.log_step('validate', row_count=1000)
report = monitor.end_run()

print('\nPipeline Report:')
for step, metrics in report['metrics'].items():
    print(f'  {step}: {metrics}')

## Summary

In this lesson, you learned:
- ETL vs ELT patterns and when to use each
- Extracting data from APIs with rate limiting and retry logic
- Working with Parquet, Avro, and Arrow data formats
- Apache Airflow concepts: DAGs, Tasks, Operators, Sensors
- Data quality validation with Great Expectations
- S3 cloud storage interaction with boto3/moto
- Pipeline monitoring and logging

**Key Practice Points:**
- Always implement rate limiting when calling external APIs
- Use Parquet for ML feature storage (compressed, columnar, fast)
- Validate data quality before feeding to ML models
- Design pipelines as DAGs with clear task dependencies
- Monitor all pipeline steps with logging and metrics